# 25. Discovering and ranking your own backends

The other notebooks target a backend by name, from the bundled calibration snapshots. This one
covers the path where you point qb-compiler at whatever hardware your account can actually reach,
and let it tell you which of those is worth the shots.

Four entry points, none of which appear elsewhere in this set:

| Function | Question it answers |
|---|---|
| `discover_backends` | What can this account see, and is it up right now? |
| `rank_discovered` | Of those, which will give the best result for *this* circuit? |
| `check_viability_pub` | Is this V2 primitives PUB worth submitting? |
| `calibration_trend` | Is this device getting better or worse? |

### Running without credentials

`discover_backends` and `rank_discovered` take a runtime service. In real use that is two lines:

```python
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService()
```

So that this notebook runs for anyone, with no account and no network, it builds a local
stand-in exposing the same surface those functions read: `backends()`, and per backend a `name`,
`num_qubits`, `target` and `status()`. Nothing below is specific to the stand-in, so swapping in
a real `QiskitRuntimeService` is the only change needed.

In [1]:
from types import SimpleNamespace

from qiskit import QuantumCircuit
from qiskit.providers.fake_provider import GenericBackendV2

from qb_compiler import (
    calibration_trend,
    check_viability_pub,
    discover_backends,
    rank_discovered,
)


class _ReportsStatus:
    """Wrap a local backend so it answers status(), as a runtime backend does.

    GenericBackendV2 has no status() method, and discovery treats a backend it cannot confirm
    is up as non-operational. Real runtime backends report status, so the wrapper supplies it.
    """

    def __init__(self, backend, pending_jobs=0):
        self._backend = backend
        self._pending_jobs = pending_jobs

    def __getattr__(self, attr):
        return getattr(self._backend, attr)

    def status(self):
        return SimpleNamespace(operational=True, pending_jobs=self._pending_jobs)


class LocalService:
    """Stand-in for QiskitRuntimeService, so the notebook needs no credentials."""

    def __init__(self):
        self._backends = [
            _ReportsStatus(GenericBackendV2(5, seed=5), pending_jobs=2),
            _ReportsStatus(GenericBackendV2(16, seed=16), pending_jobs=41),
            _ReportsStatus(GenericBackendV2(27, seed=27), pending_jobs=7),
        ]

    def backends(self):
        return self._backends


service = LocalService()
print(f"stand-in service exposing {len(service.backends())} backends")

stand-in service exposing 3 backends


## 1. What can this account reach?

`discover_backends` takes one snapshot per backend: how many qubits, whether it is operational,
how deep the queue is, and whether it exposes a transpiler target. A backend whose `status()`
call fails is recorded as non-operational rather than raising, so one unreachable device does
not abort the scan.

In [2]:
discovered = discover_backends(service)

print(f"{'backend':<24} {'qubits':>7} {'up':>5} {'queued':>7}  basis")
print("-" * 78)
for backend in discovered:
    basis = ", ".join(sorted(backend.basis_gates)[:5])
    print(
        f"{backend.name:<24} {backend.num_qubits:>7} "
        f"{str(backend.operational):>5} {backend.pending_jobs:>7}  {basis}"
    )

backend                   qubits    up  queued  basis
------------------------------------------------------------------------------
generic_backend_5q             5  True       2  cx, delay, id, measure, reset
generic_backend_16q           16  True      41  cx, delay, id, measure, reset
generic_backend_27q           27  True       7  cx, delay, id, measure, reset


## 2. Which one should run this circuit?

`discover_backends` says what exists. `rank_discovered` says which is worth using, by running a
viability check per backend against its live target and sorting on estimated fidelity.

Backends too small for the circuit, or not operational, are skipped rather than ranked, so the
result can be shorter than the discovered list. That is the intended answer to "where should
this run", not a filtered inventory.

In [3]:
ghz = QuantumCircuit(5)
ghz.h(0)
for qubit in range(4):
    ghz.cx(qubit, qubit + 1)
ghz.measure_all()

ranked = rank_discovered(ghz, service, n_seeds=2)

print(f"{'rank':<5} {'backend':<24} {'status':<10} {'est. fidelity':>14} {'queued':>7}")
print("-" * 66)
for position, (backend, viability) in enumerate(ranked, start=1):
    print(
        f"{position:<5} {backend.name:<24} {viability.status:<10} "
        f"{viability.estimated_fidelity:>14.4f} {backend.pending_jobs:>7}"
    )

skipped = len(discovered) - len(ranked)
print(f"\n{len(ranked)} ranked, {skipped} skipped as too small or not operational")

rank  backend                  status      est. fidelity  queued
------------------------------------------------------------------
1     generic_backend_5q       VIABLE             0.9321       2
2     generic_backend_16q      VIABLE             0.9321      41
3     generic_backend_27q      VIABLE             0.9321       7

3 ranked, 0 skipped as too small or not operational


## 3. Checking a PUB before submitting it

Qiskit V2 primitives take PUBs, a tuple of `(circuit, parameter_values, ...)` rather than a bare
circuit. `check_viability_pub` unpacks the circuit from the PUB and runs the same viability check,
so a submission can be assessed in the form it will actually be sent.

In [4]:
parameterised = QuantumCircuit(4)
parameterised.h(0)
for qubit in range(3):
    parameterised.cx(qubit, qubit + 1)
parameterised.measure_all()

pub = (parameterised,)
result = check_viability_pub(pub, backend="ibm_fez")

print(f"status:           {result.status}")
print(f"est. fidelity:    {result.estimated_fidelity:.4f}")
print(f"noise floor:      {result.noise_floor:.4f}")
for suggestion in result.suggestions[:3]:
    print(f"  - {suggestion}")

status:           VIABLE
est. fidelity:    0.8771
noise floor:      0.0625
  - Circuit looks good: proceed with execution.
  - Calibration snapshot is 135 days old; the estimate reflects that date. Pass fresh backend_props or set QBC_CALIBRATION_DIR for current numbers.


## 4. Is the device drifting?

Calibration is not static, and a backend that was the right choice last week may not be today.
`calibration_trend` compares the median two-qubit error across the on-disk snapshots for a
backend and reports the direction.

This reads whatever snapshots are available locally. A small set ships in the wheel so this works
from a plain install; point `QBC_CALIBRATION_DIR` at your own snapshot directory for live data.
The comparison is deliberately a plain before-and-after across the window, not a forecast.

In [5]:
for backend_name in ("ibm_fez", "ibm_torino"):
    try:
        direction, detail = calibration_trend(backend_name, window=5)
        print(f"{backend_name:<14} {direction:<12} {detail}")
    except Exception as exc:
        print(f"{backend_name:<14} no trend available ({type(exc).__name__})")

ibm_fez        improving    Median two-qubit error moved from 0.00580 to 0.00266 across 3 snapshots over 27 days; fitted change -0.00321 (-86.0 percent of the mean), so improving. Naive slope on public calibration data, signal only.
ibm_torino     unknown      Found 1 dated calibration snapshot(s) for ibm_torino; at least 2 are needed to estimate a trend.


## Summary

- `discover_backends` turns an account into an inventory, recording rather than raising when a
  device cannot be reached.
- `rank_discovered` answers which of those to use for a given circuit, and returns fewer entries
  than it discovered when some are unsuitable.
- `check_viability_pub` assesses a submission in PUB form.
- `calibration_trend` reports the direction of drift from local snapshots.

Swap `LocalService` for `QiskitRuntimeService()` and the same calls run against real hardware.